In [ ]:
%pip install transformers datasets torch -q


In [ ]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
from torch.utils.data import Dataset

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)


In [ ]:
fake_df = pd.read_csv("/content/True (1).csv")
real_df = pd.read_csv("/content/Fake (1).csv")

fake_df["label"] = 0
real_df["label"] = 1

df = pd.concat([fake_df, real_df])
df = df.sample(frac=1).reset_index(drop=True)

df["content"] = df["title"] + " " + df["text"]

print("Total Articles:", len(df))


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["content"] = df["content"].apply(clean_text)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["content"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))


In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")


In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [ ]:
train_dataset = NewsDataset(X_train, y_train)
test_dataset = NewsDataset(X_test, y_test)


In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)


In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()


In [ ]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
%pip install wikipedia


In [ ]:
import wikipedia

def retrieve_evidence(query):
    try:
        summary = wikipedia.summary(query, sentences=2)
        return summary
    except:
        return "No strong evidence found online."


In [ ]:
def bert_plus_evidence(news_text):
    inputs = tokenizer(
        news_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).detach().cpu().numpy()[0]

    fake_prob = probs[0]
    real_prob = probs[1]

    print("NEWS CLAIM:")
    print(news_text)

    print("BERT Prediction:")
    if real_prob > fake_prob:
        print("Real News Style ")
    else:
        print("Fake News Style ")

    print("Confidence Scores:")
    print("Fake Confidence:", fake_prob)
    print("Real Confidence:", real_prob)
    print("Evidence Retrieved Wikipedia:")
    evidence = retrieve_evidence(news_text[:80])
    print(evidence)
    print("Final Decision:")
    if real_prob > 0.8 and evidence != "No strong evidence found online.":
        print("Likely REAL + Evidence Exists ")
    elif fake_prob > 0.8 and evidence == "No strong evidence found online.":
        print("Likely FAKE and No Evidence ")
    else:
        print("News is uncertain  Needs Verification ")

In [ ]:
bert_plus_evidence("Babar boost for Pakistan but Bangladesh have all the momentum")